In [ ]:
# =====================================================
# DOWNSTREAM EVALUATION ON HELD-OUT SPLITS (multi-baseline)
# - Canonicalize/merge label variants
# - Evaluate many embedding sets with linear probe & k-NN
# - Extras: Few-shot curves & k-NN sensitivity
# - Save per-model results + combined CSV + plots
# - NEW: Skip guards to avoid recomputing existing baselines
# =====================================================

import os, copy, re, glob, json
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import ParameterGrid

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

SEED = 6740
np.random.seed(SEED)

# -------------------------
# Speed / behavior knobs
# -------------------------
FAST_MODE = True                       # flip False for full evaluation
FEW_SHOT_GRID = [1, 5, 10, 25, None]   # None means "All train"
K_GRID = [1, 5, 10, 20, 50]
K_FOR_KNN = 10 if FAST_MODE else 20
C_GRID = [0.1, 1.0] if FAST_MODE else [0.01, 0.1, 0.5, 1.0, 2.0, 5.0]
MAX_ITER_LR = 800 if FAST_MODE else 2000
MIN_CLASS_COUNT = 100 if FAST_MODE else 50
PCA_DIM = None                         # e.g., 256 to speed up; None disables
METABOFM_VERSION = "last"

# -------------------------
# Skips & guards
# -------------------------
ONLY_EVAL_EXISTING = True     # only run models that already have embeddings saved
SKIP_IF_RESULTS_EXIST = True  # don't recompute per-model results if CSVs already exist
FORCE_REEVAL = False          # set True to ignore SKIP_IF_RESULTS_EXIST

FEATS_PRIMARY = "image_feats.npy"
FEATS_ALT_LAST = "image_feats_last.npy"
INDEX_FILE = "index.csv"

# keep a simple constant for the index
REQUIRED_INDEX = INDEX_FILE

def get_feats_path(model_tag: str, emb_dir: str) -> str | None:
    """Return the feature path for the given model."""
    if model_tag == "metabofm":
        fname = FEATS_PRIMARY if METABOFM_VERSION == "best" else FEATS_ALT_LAST
        path = os.path.join(emb_dir, fname)
        if os.path.exists(path):
            print(f"[INFO] Using MetaboFM-{METABOFM_VERSION.upper()} embeddings: {path}")
            return path
        else:
            print(f"[WARN] MetaboFM-{METABOFM_VERSION.upper()} embeddings not found at {path}")
            return None
    else:
        path = os.path.join(emb_dir, FEATS_PRIMARY)
        return path if os.path.exists(path) else None

def embeddings_ready(model_tag: str, emb_dir: str) -> bool:
    feats_path = get_feats_path(model_tag, emb_dir)
    return feats_path is not None and os.path.exists(os.path.join(emb_dir, REQUIRED_INDEX))

def resolve_model_outdir(model_tag: str, emb_dir: str) -> str:
    if model_tag == "metabofm":
        return TRAIN_OUT
    return emb_dir

def results_already_done(model_tag: str, emb_dir: str) -> bool:
    primary_out_dir = resolve_model_outdir(model_tag, emb_dir)
    res_csv = os.path.join(primary_out_dir, "downstream_results.csv")
    return os.path.exists(res_csv)

def inputs_newer_than_results(model_tag: str, emb_dir: str) -> bool:
    """If inputs are newer than results, you may want to recompute."""
    primary_out_dir = resolve_model_outdir(model_tag, emb_dir)
    res_csv = os.path.join(primary_out_dir, "downstream_results.csv")
    if not os.path.exists(res_csv):
        return True
    res_mtime = os.path.getmtime(res_csv)
    latest_input_mtime = max(
        os.path.getmtime(os.path.join(emb_dir, f))
        for f in REQUIRED_FILES
        if os.path.exists(os.path.join(emb_dir, f))
    )
    return latest_input_mtime > res_mtime

def inputs_newer_than_results(model_tag: str, emb_dir: str) -> bool:
    primary_out_dir = resolve_model_outdir(model_tag, emb_dir)
    res_csv = os.path.join(primary_out_dir, "downstream_results.csv")
    if not os.path.exists(res_csv):
        return True
    res_mtime = os.path.getmtime(res_csv)

    feats_path = get_feats_path(model_tag, emb_dir)
    idx_path = os.path.join(emb_dir, REQUIRED_INDEX)
    inputs = [p for p in [feats_path, idx_path] if p and os.path.exists(p)]
    if not inputs:
        return True
    latest_input_mtime = max(os.path.getmtime(p) for p in inputs)
    return latest_input_mtime > res_mtime

# -------------------------
# Paths / run root
# -------------------------

# Metadata + splits
SPLIT_CSV    = os.path.join("splits_by_dataset_id.csv")
IDX_PARQUET  = "metaspace_images_dump/msi_fm_samples.parquet"
MAN_PARQUET  = "metaspace_images_dump/manifest_expanded.parquet"

# Centralized benchmark outputs
BENCH_OUT = os.path.join("baseline_eval")
PLOTS_OUT = os.path.join(BENCH_OUT, "plots")
os.makedirs(BENCH_OUT, exist_ok=True)
os.makedirs(PLOTS_OUT, exist_ok=True)

# -------------------------
# Which embedding sets to evaluate
# (ensure these folders contain image_feats.npy + index.csv)
# -------------------------
EMB_SETS = {
    "msi_multitask_dinov2_untrained":    os.path.join("pretrained_feats", "msi_multitask_dinov2_untrained"),
    "msi_multitask_mae_untrained":    os.path.join("pretrained_feats", "msi_multitask_mae_untrained"),
    "msi_multitask_dinov2":    os.path.join("pretrained_feats", "msi_multitask_dinov2"),
    "msi_multitask_mae":    os.path.join("pretrained_feats", "msi_multitask_mae")
}

# -------------------------
# Canonicalization / merging rules
# -------------------------
def _clean(s):
    if pd.isna(s): return None
    s = str(s).strip()
    s = re.sub(r"\s+", " ", s)
    return s

def canonicalize_labels(df):
    df = df.copy()

    # 1) Polarity
    pol_map = {"pos":"Positive","positive":"Positive","+":"Positive",
               "neg":"Negative","negative":"Negative","-":"Negative"}
    def canon_polarity(s):
        if s is None: return None
        t = _clean(s).lower()
        t2 = pol_map.get(t, t)
        if t2 in ("positive","negative"):
            return t2.capitalize()
        if "pos" in t: return "Positive"
        if "neg" in t: return "Negative"
        return _clean(s)
    if "polarity" in df.columns:
        df["polarity"] = df["polarity"].map(canon_polarity)

    # 2) Ionisation Source (map DESI-MSI -> DESI, etc.)
    def canon_ion_src(s):
        if s is None: return None
        t_raw = _clean(s)
        t = t_raw.upper().replace("-", "").replace("_","")
        if "APSMALDI" in t: return "AP-SMALDI"
        if "IRMALDESI" in t or "IRMALDI" in t: return "IR-MALDESI"
        if "APMALDI" in t: return "AP-MALDI"
        if "DESIMSI" in t: return "DESI"
        if "DESI" in t: return "DESI"
        if "MALDI" in t: return "MALDI"
        return t_raw
    if "ionisationSource" in df.columns:
        df["ionisationSource"] = df["ionisationSource"].map(canon_ion_src)

    # 3) Analyzer Type
    def canon_analyzer(s):
        if s is None: return None
        t = _clean(s); tl = t.lower()
        if "timstof" in tl and "flex" in tl: return "timsTOF Flex"
        if "fticr" in tl:
            if "12t" in tl: return "12T FTICR"
            if "7t" in tl and "scimax" in tl: return "FTICR scimaX 7T"
            return "FTICR"
        if "orbitrap" in tl or "q-exactive" in tl: return "Orbitrap"
        if "tof" in tl and "reflector" in tl: return "TOF reflector"
        if tl.strip() == "qtof": return "qTOF"
        return t
    if "analyzerType" in df.columns:
        df["analyzerType"] = df["analyzerType"].map(canon_analyzer)

    # 4) Organism
    def canon_organism(s):
        if s is None: return None
        t = _clean(s); tl = t.lower()
        if "|" in t or "," in t:
            if ("human" in tl or "homo sapiens" in tl) and ("mouse" in tl or "mus musculus" in tl):
                return "Mixed"
        if "homo sapiens" in tl or tl.strip() in {"human","h. sapiens","homo"}:
            return "Homo sapiens"
        if "mus musculus" in tl or tl.strip() in {"mouse","m. musculus"}:
            return "Mus musculus"
        return t
    if "organism" in df.columns:
        df["organism"] = df["organism"].map(canon_organism)

    # 5) Organism_Part
    def canon_part(s):
        if s is None: return None
        t = _clean(s); tl = t.lower()
        if "kidney" in tl: return "Kidney"
        if "brain"  in tl: return "Brain"
        if "liver"  in tl: return "Liver"
        if "lung"   in tl: return "Lung"
        if "breast" in tl: return "Breast"
        if "skin"   in tl: return "Skin"
        if "heart"  in tl or "cardiac" in tl: return "Heart"
        return t
    if "Organism_Part" in df.columns:
        df["Organism_Part"] = df["Organism_Part"].map(canon_part)

    # 6) Condition (keep "NA" but we'll exclude it later)
    def canon_condition(s):
        if s is None: return None
        t = _clean(s); tl = t.lower()
        if tl in {"n/a","na","none","not available",""}: return "NA"
        if tl in {"biopsy","biopsies"}: return "Biopsy"
        if "fresh frozen" in tl or "frozen" in tl: return "Frozen"
        if "tumor" in tl or "tumour" in tl: return "Tumor"
        if "cancer" in tl: return "Cancer"
        if "wildtype" in tl or tl == "wt": return "Wildtype"
        if "healthy" in tl or "control" in tl: return "Healthy"
        if "diseased" in tl or "disease" in tl: return "Diseased"
        return t
    if "Condition" in df.columns:
        df["Condition"] = df["Condition"].map(canon_condition)

    return df

# -------------------------
# Load metadata + splits (once)
# -------------------------
idx = pd.read_parquet(IDX_PARQUET)
man = pd.read_parquet(MAN_PARQUET)

need_cols = [
    "dataset_id", "organism", "polarity", "Organism_Part", "Condition",
    "analyzerType", "ionisationSource"
]
man_sub = man[[c for c in need_cols if c in man.columns]].drop_duplicates("dataset_id")

df_meta = idx.merge(man_sub, on="dataset_id", how="left", suffixes=("", "_man"))
df_meta = df_meta.loc[:, ~df_meta.columns.duplicated()].copy().reset_index(drop=True)
splits = pd.read_csv(SPLIT_CSV)
df_meta = df_meta.merge(splits, on="dataset_id", how="left")

# Dedup metadata by sample_path and canonicalize labels
if df_meta.duplicated("sample_path").sum():
    print("[WARN] duplicate sample_path rows in metadata; keeping first.")
    df_meta = df_meta.drop_duplicates("sample_path", keep="first").reset_index(drop=True)
df_meta = canonicalize_labels(df_meta)

# -------------------------
# Common helpers
# -------------------------
TASKS = ["organism", "polarity", "Organism_Part", "Condition", "analyzerType", "ionisationSource"]
EXCLUDE_LABELS = {"Condition": {"NA"}}

def filter_valid(df_task, yname, min_count=5):
    x = df_task.dropna(subset=[yname]).copy()
    if yname in EXCLUDE_LABELS:
        x = x[~x[yname].isin(EXCLUDE_LABELS[yname])]
    x = x[x[yname].astype(str).str.len() > 0]
    vc = x[yname].value_counts()
    keep = vc[vc >= min_count].index
    x = x[x[yname].isin(keep)].copy()
    return x

def few_shot_subset(df_task, yname, shots_per_class=None, seed=SEED):
    if not shots_per_class or shots_per_class <= 0:
        return (df_task["split"] == "train").values
    rng = np.random.RandomState(seed)
    m_train = (df_task["split"] == "train").values
    keep = np.zeros(len(df_task), dtype=bool)
    labels = df_task.loc[m_train, yname].astype(str).values
    idxs   = np.where(m_train)[0]
    from collections import defaultdict
    per_class = defaultdict(list)
    for i, lbl in zip(idxs, labels):
        per_class[lbl].append(i)
    for lbl, arr in per_class.items():
        arr = np.array(arr)
        rng.shuffle(arr)
        keep[arr[:min(shots_per_class, len(arr))]] = True
    return keep

def get_mask(df_all, split_name):
    return (df_all["split"] == split_name).values

def run_linear_probe(X_tr, y_tr, X_va, y_va, X_te, y_te):
    pipe = Pipeline([
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
        ("clf", LogisticRegression(
            solver="saga",
            max_iter=MAX_ITER_LR,
            class_weight="balanced",
            random_state=SEED,
            n_jobs=-1
        ))
    ])
    grid = {"clf__C": C_GRID}
    best = None; best_va = -np.inf
    for p in tqdm(list(ParameterGrid(grid)), desc="LinearProbe grid", leave=False):
        pipe.set_params(**p)
        pipe.fit(X_tr, y_tr)
        pred_va = pipe.predict(X_va)
        macro_f1 = f1_score(y_va, pred_va, average="macro")
        if macro_f1 > best_va:
            best_va = macro_f1
            best = copy.deepcopy(pipe)
    pred_te = best.predict(X_te)
    acc = accuracy_score(y_te, pred_te)
    f1m = f1_score(y_te, pred_te, average="macro")
    return acc, f1m, pred_te, best

def run_knn(X_tr, y_tr, X_te, y_te, k=20):
    n_fit = int(X_tr.shape[0])
    # need at least 1 sample and 2 classes to classify
    if n_fit < 1 or len(np.unique(y_tr)) < 2:
        return np.nan, np.nan, np.array([], dtype=object), None, 0

    k_eff = max(1, min(k, n_fit))
    knn = KNeighborsClassifier(n_neighbors=k_eff, metric="cosine", n_jobs=-1)
    knn.fit(X_tr, y_tr)

    if X_te.shape[0] == 0:
        return np.nan, np.nan, np.array([], dtype=object), knn, k_eff

    pred_te = knn.predict(X_te)
    acc = accuracy_score(y_te, pred_te)
    f1m = f1_score(y_te, pred_te, average="macro")
    return acc, f1m, pred_te, knn, k_eff

def per_class_f1(y_true, y_pred):
    rep = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    out = {k: v["f1-score"] for k, v in rep.items() if k not in ("accuracy","macro avg","weighted avg")}
    return out

# -------------------------
# Evaluate ONE embedding set dir
# -------------------------
def evaluate_embeddings(emb_dir: str, model_tag: str):
    feats_path = get_feats_path(model_tag, emb_dir)
    index_path = os.path.join(emb_dir, INDEX_FILE)
    if not (feats_path and os.path.exists(feats_path) and os.path.exists(index_path)):
        print(f"[WARN] Missing feats or index for {model_tag} at {emb_dir}; "
              f"looked for {FEATS_PRIMARY}"
              f"{' or ' + FEATS_ALT_LAST if model_tag=='metabofm' else ''} and {INDEX_FILE}. Skipping.")
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    # Optional: let the logs show which file was used
    if os.path.basename(feats_path) == FEATS_ALT_LAST:
        print(f"[INFO] ({model_tag}) Using LAST embeddings: {feats_path}")
    else:
        print(f"[INFO] ({model_tag}) Using BEST embeddings: {feats_path}")

    emb = np.load(feats_path)
    index = pd.read_csv(index_path)      # must include column 'sample_path'
    index = index.reset_index().rename(columns={"index": "row_id"})

    # Join metadata to index
    df_base = df_meta.copy()
    df_emb = df_base.merge(index, on="sample_path", how="inner")
    df_emb = df_emb.sort_values("row_id").reset_index(drop=True)

    if emb.shape[0] != len(df_emb):
        print(f"[INFO] ({model_tag}) Embedding count != joined rows; aligning to matched rows only.")
        emb = emb[df_emb["row_id"].values, :]
    if emb.shape[0] != len(df_emb):
        raise RuntimeError(f"({model_tag}) Embedding count and joined rows mismatch after alignment.")

    # Optional PCA
    if PCA_DIM is not None and PCA_DIM > 0 and PCA_DIM < emb.shape[1]:
        print(f"[INFO] ({model_tag}) Reducing dim to {PCA_DIM} via PCA.")
        from sklearn.decomposition import PCA
        pca = PCA(n_components=PCA_DIM, random_state=SEED)
        emb = pca.fit_transform(emb)

    df_emb["row_pos"] = np.arange(len(df_emb), dtype=int)

    # ---- Main evaluation (All-train or FEW_SHOT_TRAIN) ----
    results = []
    pcs_rows = []   # per-class F1 (linear probe) rows
    kgrid_rows = [] # k-sweep rows
    for yname in tqdm(TASKS, desc=f"Tasks[{model_tag}]"):
        if yname not in df_emb.columns:
            print(f"[WARN] ({model_tag}) Missing column {yname}; skipping.")
            continue

        print(f"\n==== [{model_tag}] Task: {yname} ====")
        df_task = filter_valid(df_emb, yname, min_count=MIN_CLASS_COUNT)
        if df_task.empty:
            print(f"[WARN] ({model_tag}) Skipping {yname}: no data after filtering.")
            continue
        if df_task["split"].isna().any():
            df_task = df_task[~df_task["split"].isna()].copy()

        row_pos = df_task["row_pos"].values
        X = emb[row_pos]
        y = df_task[yname].astype(str).values
        m_tr_all = get_mask(df_task, "train")
        m_va     = get_mask(df_task, "val")
        m_te     = get_mask(df_task, "test")

        def _ok(mask):
            return np.sum(mask) > 0 and (len(np.unique(y[mask])) > 1)

        # ---- Few-shot sweep (uses val for C tuning; test for final) ----
        for shots in FEW_SHOT_GRID:
            m_tr = few_shot_subset(df_task, yname, shots_per_class=shots)
            if not (_ok(m_tr) and _ok(m_va) and _ok(m_te)):
                print(f"[WARN] ({model_tag}) {yname} few-shot={shots}: insufficient classes.")
                continue

            # Linear probe (C grid on val)
            acc_lp, f1_lp, pred_lp, best_lp = run_linear_probe(
                X[m_tr], y[m_tr], X[m_va], y[m_va], X[m_te], y[m_te]
            )
            # k-NN with default K_FOR_KNN
            acc_knn, f1_knn, pred_knn, _, k_eff = run_knn(
                X[m_tr], y[m_tr], X[m_te], y[m_te], k=K_FOR_KNN
            )

            results.append({
                "model": model_tag,
                "task": yname,
                "shots_per_class": shots if shots else 0,
                "test_acc_linear": acc_lp,
                "test_macroF1_linear": f1_lp,
                "test_acc_knn": acc_knn,
                "test_macroF1_knn": f1_knn,
                "k_for_knn": int(k_eff),
                "pca_dim": PCA_DIM if PCA_DIM else X.shape[1],
                "n_train": int(m_tr.sum()),
                "n_val": int(m_va.sum()),
                "n_test": int(m_te.sum()),
                "n_classes": int(len(np.unique(y)))
            })

            # Per-class F1 (only for All-train to keep size modest)
            if shots is None:
                pcs = per_class_f1(y[m_te], pred_lp)
                for cls, f1v in pcs.items():
                    pcs_rows.append({
                        "model": model_tag,
                        "task": yname,
                        "label": cls,
                        "f1": f1v,
                        "shots_per_class": 0  # 0 = All-train
                    })

            # k-sweep on All-train only
            if shots is None:
                for k in K_GRID:
                    acc_k, f1_k, _, _, k_eff = run_knn(X[m_tr], y[m_tr], X[m_te], y[m_te], k=k)
                    if np.isnan(f1_k):
                        continue
                    kgrid_rows.append({
                        "model": model_tag,
                        "task": yname,
                        "k": int(k_eff),
                        "test_macroF1_knn": f1_k
                    })

    res_df = pd.DataFrame(results).sort_values(["task", "model", "shots_per_class"])
    pcs_df = pd.DataFrame(pcs_rows).sort_values(["task", "model", "label"])
    kgrid_df = pd.DataFrame(kgrid_rows).sort_values(["task", "model", "k"])

    # --------- SAVE: per-model to its corresponding location ---------
    primary_out_dir = resolve_model_outdir(model_tag, emb_dir)
    os.makedirs(primary_out_dir, exist_ok=True)

    res_path = os.path.join(primary_out_dir, "downstream_results.csv")
    res_df.to_csv(res_path, index=False)
    if len(pcs_df):
        pcs_path = os.path.join(primary_out_dir, "per_class_f1_linear.csv")
        pcs_df.to_csv(pcs_path, index=False)
    if len(kgrid_df):
        ks_path = os.path.join(primary_out_dir, "knn_k_sweep.csv")
        kgrid_df.to_csv(ks_path, index=False)

    print(f"[OK] ({model_tag}) Saved per-model results to: {primary_out_dir}")

    # Also mirror into BENCH_OUT/model_tag for centralized copy
    mirror_dir = os.path.join(BENCH_OUT, model_tag)
    os.makedirs(mirror_dir, exist_ok=True)
    res_df.to_csv(os.path.join(mirror_dir, "downstream_results.csv"), index=False)
    if len(pcs_df):
        pcs_df.to_csv(os.path.join(mirror_dir, "per_class_f1_linear.csv"), index=False)
    if len(kgrid_df):
        kgrid_df.to_csv(os.path.join(mirror_dir, "knn_k_sweep.csv"), index=False)

    return res_df, pcs_df, kgrid_df

# -------------------------
# Run all baselines + FM and aggregate (with skip guards)
# -------------------------
all_main, all_pcs, all_ks = [], [], []
for tag, emb_dir in EMB_SETS.items():
    # 1) Only run if embeddings are present (for baselines)
    if ONLY_EVAL_EXISTING and not embeddings_ready(tag, emb_dir):
        print(f"[SKIP] '{tag}': embeddings missing at {emb_dir} "
            f"(need {FEATS_PRIMARY} or {FEATS_ALT_LAST} for metabofm, plus {INDEX_FILE}); skipping.")
        continue


    # 2) Skip if results already exist (unless inputs are newer or you force)
    if SKIP_IF_RESULTS_EXIST and results_already_done(tag, emb_dir) and not FORCE_REEVAL:
        if inputs_newer_than_results(tag, emb_dir):
            print(f"[RERUN] '{tag}': inputs newer than results; re-evaluating.")
        else:
            print(f"[SKIP] '{tag}': results already exist and are up-to-date.")
            # Load existing and append to aggregates (and ensure mirror exists)
            try:
                primary_out_dir = resolve_model_outdir(tag, emb_dir)
                res_df = pd.read_csv(os.path.join(primary_out_dir, "downstream_results.csv"))
                all_main.append(res_df)

                pcs_path = os.path.join(primary_out_dir, "per_class_f1_linear.csv")
                if os.path.exists(pcs_path):
                    all_pcs.append(pd.read_csv(pcs_path))
                ks_path = os.path.join(primary_out_dir, "knn_k_sweep.csv")
                if os.path.exists(ks_path):
                    all_ks.append(pd.read_csv(ks_path))

                # Mirror to BENCH_OUT/tag if not already mirrored
                mirror_dir = os.path.join(BENCH_OUT, tag)
                os.makedirs(mirror_dir, exist_ok=True)
                res_df.to_csv(os.path.join(mirror_dir, "downstream_results.csv"), index=False)
                if os.path.exists(pcs_path):
                    pd.read_csv(pcs_path).to_csv(os.path.join(mirror_dir, "per_class_f1_linear.csv"), index=False)
                if os.path.exists(ks_path):
                    pd.read_csv(ks_path).to_csv(os.path.join(mirror_dir, "knn_k_sweep.csv"), index=False)
            except Exception as e:
                print(f"[WARN] Failed to load existing results for '{tag}': {e}")
            continue

    print(f"\n[RUN] Evaluating model '{tag}' from {emb_dir}")
    res_df, pcs_df, kgrid_df = evaluate_embeddings(emb_dir, tag)
    if len(res_df):  all_main.append(res_df)
    if len(pcs_df):  all_pcs.append(pcs_df)
    if len(kgrid_df): all_ks.append(kgrid_df)

### Few-shot Evaluation

In [ ]:
# =====================================================
# DOWNSTREAM EVALUATION ON HELD-OUT SPLITS (multi-baseline)
# - Canonicalize/merge label variants
# - Evaluate many embedding sets with linear probe & k-NN
# - Extras: Few-shot curves, k-NN sensitivity, label-efficiency AUC, win-rate, k-robustness
# - Save per-model results + centralized mirrors + combined plots
# - Skip guards to avoid recomputing existing baselines
# - NEW: Save CSV backing each plot
# =====================================================

import os, copy, re, glob, json, hashlib
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import ParameterGrid

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import matplotlib as mpl
mpl.rcParams['svg.fonttype'] = 'none'

SEED = 6740
np.random.seed(SEED)

# -------------------------
# Speed / behavior knobs
# -------------------------
FAST_MODE = True                       # flip False for full evaluation
FEW_SHOT_GRID = [1, 5, 10, 25, None]   # None means "All train"
K_GRID = [1, 5, 10, 20, 50]
K_FOR_KNN = 10 if FAST_MODE else 20
C_GRID = [0.1, 1.0] if FAST_MODE else [0.01, 0.1, 0.5, 1.0, 2.0, 5.0]
MAX_ITER_LR = 800 if FAST_MODE else 2000
MIN_CLASS_COUNT = 100 if FAST_MODE else 50
PCA_DIM = None                         # e.g., 256 to speed up; None disables
METABOFM_VERSION = "last"              # "best" or "last"

# -------------------------
# Skips & guards
# -------------------------
ONLY_EVAL_EXISTING = True     # only run models that already have embeddings saved
SKIP_IF_RESULTS_EXIST = True  # don't recompute per-model results if CSVs already exist
FORCE_REEVAL = False          # set True to ignore SKIP_IF_RESULTS_EXIST

FEATS_PRIMARY = "image_feats.npy"
FEATS_ALT_LAST = "image_feats_last.npy"
INDEX_FILE = "index.csv"
REQUIRED_INDEX = INDEX_FILE

# -------------------------
# Config (match your eval script knobs)
# -------------------------

COMBINED_BENCH_OUT = os.path.join("baseline_eval_combined")
COMBINED_PLOTS_OUT = os.path.join(COMBINED_BENCH_OUT, "plots")
os.makedirs(COMBINED_PLOTS_OUT, exist_ok=True)

# Keep same task list
TASKS = ["organism", "polarity", "Organism_Part", "Condition", "analyzerType", "ionisationSource"]

# Off-the-shelf baselines to include in combined plots
BASELINE_EMB_SETS = {
    "MAE-ViT-B/16 – Pretrained":    os.path.join("pretrained_feats", "msi_multitask_mae_untrained"),
    "DINOv2-ViT-B/14 – Pretrained":    os.path.join("pretrained_feats", "msi_multitask_dinov2_untrained"),
    "MAE-ViT-B/16 – Fine-tuned":    os.path.join("pretrained_feats", "msi_multitask_mae"),
    "DINOv2-ViT-B/14 – Fine-tuned":    os.path.join("pretrained_feats", "msi_multitask_dinov2"),
}

# -------------------------
# Helpers
# -------------------------
def save_df_csv(df: pd.DataFrame, path: str):
    """Safely save dataframe to CSV (and ensure dir)."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    df.to_csv(path, index=False)

def load_first_existing_csv(paths):
    for p in paths:
        if p and os.path.exists(p):
            return pd.read_csv(p), p
    return None, None

def read_vit_name_from_run(run_dir: str) -> str:
    rp = os.path.join(run_dir, "run_params.json")
    if os.path.exists(rp):
        try:
            with open(rp, "r", encoding="utf-8") as f:
                data = json.load(f)
            if "cfg" in data and isinstance(data["cfg"], dict):
                vit = data["cfg"].get("vit_name", None)
                if vit:
                    return str(vit)
            vit = data.get("vit_name", None)
            if vit:
                return str(vit)
        except Exception as e:
            print(f"[WARN] Failed reading vit_name from {rp}: {e}")
    return os.path.basename(os.path.normpath(run_dir))

def make_unique_name(base_name: str, existing: set, disambig_hint: str) -> str:
    name = base_name
    if name not in existing:
        return name
    short = disambig_hint.replace("\\", "/").strip("/").split("/")[-1]
    suffix = f" ({short})"
    if name + suffix not in existing:
        return name + suffix
    tiny = hashlib.md5(disambig_hint.encode("utf-8")).hexdigest()[:6]
    alt = f"{name} [{tiny}]"
    if alt not in existing:
        return alt
    i = 2
    while True:
        cand = f"{alt}-{i}"
        if cand not in existing:
            return cand
        i += 1

def primary_results_dir_for_fm_run(run_dir: str) -> str:
    return run_dir

def primary_results_dir_for_baseline(baseline_emb_dir: str) -> str:
    return baseline_emb_dir

# -------------------------
# Build the model list (FM runs + baselines)
# -------------------------
model_entries = []
seen_names = set()

# Add baselines
for tag, emb_dir in BASELINE_EMB_SETS.items():
    disp = make_unique_name(tag, seen_names, disambig_hint=emb_dir)
    seen_names.add(disp)
    model_entries.append({
        "name": disp,
        "kind": "baseline",
        "primary_dir": primary_results_dir_for_baseline(emb_dir),
        "mirror_dir": os.path.join(COMBINED_BENCH_OUT, disp.replace(os.sep, "_")),
    })

# -------------------------
# Load per-model CSVs
# -------------------------
all_main, all_pcs, all_ks = [], [], []
loaded = []

for m in model_entries:
    name = m["name"]
    primary_dir = m["primary_dir"]
    mirror_dir  = m["mirror_dir"]

    main_paths = [
        os.path.join(primary_dir, "downstream_results.csv"),
        os.path.join(mirror_dir,  "downstream_results.csv"),
    ]
    pcs_paths = [
        os.path.join(primary_dir, "per_class_f1_linear.csv"),
        os.path.join(mirror_dir,  "per_class_f1_linear.csv"),
    ]
    ks_paths = [
        os.path.join(primary_dir, "knn_k_sweep.csv"),
        os.path.join(mirror_dir,  "knn_k_sweep.csv"),
    ]

    df_main, mp = load_first_existing_csv(main_paths)
    if df_main is None:
        print(f"[WARN] No downstream_results.csv for '{name}'. Skipping this model.")
        continue

    df_pcs, pp = load_first_existing_csv(pcs_paths)
    df_ks,  kp = load_first_existing_csv(ks_paths)

    required_cols = {"model","task","shots_per_class","test_macroF1_linear","test_macroF1_knn"}
    missing = required_cols.difference(set(df_main.columns) | {"model"})  # model will be added below
    if missing:
        raise ValueError(f"[ERR] '{name}' main CSV missing columns: {missing} at {mp}")

    df_main = df_main.copy()
    df_main["model"] = name

    all_main.append(df_main)
    if df_pcs is not None:
        df_pcs = df_pcs.copy()
        df_pcs["model"] = name
        all_pcs.append(df_pcs)
    if df_ks is not None:
        if "k" not in df_ks.columns or "test_macroF1_knn" not in df_ks.columns:
            print(f"[WARN] k-sweep CSV for '{name}' missing expected columns; skipping.")
        else:
            df_ks = df_ks.copy()
            df_ks["model"] = name
            all_ks.append(df_ks)

    loaded.append((name, mp, pp, kp))

if not all_main:
    raise SystemExit("[ERR] No per-model CSVs could be loaded. Check FM_RUN_DIRS and baseline paths.")

bench   = pd.concat(all_main, ignore_index=True)
pcs_all = pd.concat(all_pcs, ignore_index=True) if all_pcs else pd.DataFrame()
ks_all  = pd.concat(all_ks,  ignore_index=True) if all_ks  else pd.DataFrame()

# Save the unified CSVs too
save_df_csv(bench,   os.path.join(COMBINED_PLOTS_OUT, "ALL_downstream_results_merged.csv"))
if len(pcs_all):
    save_df_csv(pcs_all, os.path.join(COMBINED_PLOTS_OUT, "ALL_per_class_f1_linear_merged.csv"))
if len(ks_all):
    save_df_csv(ks_all,  os.path.join(COMBINED_PLOTS_OUT, "ALL_knn_k_sweep_merged.csv"))

print("[OK] Loaded models:")
for name, mp, pp, kp in loaded:
    print(f" - {name}\n    main: {mp}\n    pcs : {pp or '—'}\n    k   : {kp or '—'}")

# -------------------------
# PLOTS (standard + narrative-pivot extras)
# -------------------------
sns.set(style="whitegrid")

# === Font knobs ===
TITLE_FS   = 18
LABEL_FS   = 16
TICK_FS    = 14
LEGEND_FS  = 13
LEGEND_TTL = 13

def _bolden_axis(ax):
    """Make titles, axis labels, and tick labels bigger + bold."""
    # Title
    if ax.get_title():
        ax.set_title(ax.get_title(), fontsize=TITLE_FS, fontweight="bold")
    # Axis labels
    if ax.get_xlabel():
        ax.set_xlabel(ax.get_xlabel(), fontsize=LABEL_FS, fontweight="bold")
    if ax.get_ylabel():
        ax.set_ylabel(ax.get_ylabel(), fontsize=LABEL_FS, fontweight="bold")
    # Tick labels
    ax.tick_params(axis="x", labelsize=TICK_FS)
    ax.tick_params(axis="y", labelsize=TICK_FS)
    plt.setp(ax.get_xticklabels(), fontweight="bold")
    plt.setp(ax.get_yticklabels(), fontweight="bold")
    # Legend (if present)
    leg = ax.get_legend()
    if leg is not None:
        plt.setp(leg.get_texts(), fontsize=LEGEND_FS, fontweight="bold")
        if leg.get_title() is not None:
            leg.get_title().set_fontsize(LEGEND_TTL)
            leg.get_title().set_fontweight("bold")

# 1) Macro-F1 by task & model (Linear Probe) — barplot (All-train only)
df_bar = bench[bench["shots_per_class"] == 0].copy()
if len(df_bar):
    # Save raw plot data
    save_df_csv(df_bar, os.path.join(COMBINED_PLOTS_OUT, "PLOT_DATA_bar_macroF1_linear_alltrain_raw.csv"))

    plt.figure(figsize=(14, 6))
    order_tasks = [t for t in TASKS if t in df_bar["task"].unique()]
    ax = sns.barplot(
        data=df_bar, x="task", y="test_macroF1_linear", hue="model",
        order=order_tasks, errorbar="ci", dodge=True
    )
    ax.set_title("Macro-F1 (Linear Probe, TEST) by Task & Model (All-train)")
    ax.set_ylabel("Macro-F1")
    ax.set_xlabel("Task")
    plt.xticks(rotation=30, ha="right")

    # Move legend and style it
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="Model", frameon=False)

    _bolden_axis(ax)
    plt.tight_layout()
    plt.savefig(os.path.join(COMBINED_PLOTS_OUT, "bar_macroF1_linear_alltrain.png"), dpi=200)
    plt.savefig(os.path.join(COMBINED_PLOTS_OUT, "bar_macroF1_linear_alltrain.svg"), format='svg', dpi=200)
    plt.close()

# 2) Macro-F1 by task & model (k-NN default K) — barplot (All-train only)
if len(df_bar):
    # Save raw plot data
    save_df_csv(df_bar, os.path.join(COMBINED_PLOTS_OUT, f"PLOT_DATA_bar_macroF1_knn_k{K_FOR_KNN}_alltrain_raw.csv"))

    plt.figure(figsize=(14, 6))
    order_tasks = [t for t in TASKS if t in df_bar["task"].unique()]
    ax = sns.barplot(
        data=df_bar, x="task", y="test_macroF1_knn", hue="model",
        order=order_tasks, errorbar="ci", dodge=True
    )
    ax.set_title(f"Macro-F1 (k-NN@{K_FOR_KNN}, TEST) by Task & Model (All-train)")
    ax.set_ylabel("Macro-F1")
    ax.set_xlabel("Task")
    plt.xticks(rotation=30, ha="right")

    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="Model", frameon=False)

    _bolden_axis(ax)
    plt.tight_layout()
    plt.savefig(os.path.join(COMBINED_PLOTS_OUT, f"bar_macroF1_knn_k{K_FOR_KNN}_alltrain.png"), dpi=200)
    plt.savefig(os.path.join(COMBINED_PLOTS_OUT, f"bar_macroF1_knn_k{K_FOR_KNN}_alltrain.svg"), format='svg', dpi=200)
    plt.close()

# ============================================
# Few-shot BARPLOTS per task (shots on x-axis)
# ============================================

# Metrics to plot: linear probe & k-NN (macro-F1 on TEST)
METRICS = [
    ("test_macroF1_linear", "Linear Probe", "linear"),
    ("test_macroF1_knn",    "k-NN",         "knn"),
]

df_fs = bench.copy()
if len(df_fs):
    # Map shots_per_class (0 means All-train)
    df_fs["shots_lbl"] = df_fs["shots_per_class"].replace({0: "All"}).astype(str)

    # Desired order of bars on the x-axis
    desired_order = ["1", "5", "10", "25", "All"]

    # Save the global few-shot merged dataframe used for all plots
    save_df_csv(df_fs, os.path.join(COMBINED_PLOTS_OUT, "PLOT_DATA_bar_fewshot_ALL_metrics_RAW.csv"))

    for metric_col, metric_title, metric_key in METRICS:
        if metric_col not in df_fs.columns:
            print(f"[WARN] Metric '{metric_col}' missing in bench; skipping all tasks for this metric.")
            continue

        for tsk in TASKS:
            cur = df_fs[df_fs["task"] == tsk].copy()
            if cur.empty:
                continue

            # Keep only present shot labels but preserve canonical order
            present = [lab for lab in desired_order if lab in set(cur["shots_lbl"])]
            if not present:
                print(f"[WARN] No few-shot settings present for task '{tsk}'. Skipping.")
                continue

            cur["shots_lbl"] = pd.Categorical(cur["shots_lbl"], categories=present, ordered=True)

            # Save per-task/metric plot data
            out_csv = os.path.join(COMBINED_PLOTS_OUT, f"PLOT_DATA_bar_fewshot_{metric_key}_{tsk}.csv")
            save_df_csv(cur[["model","task","shots_per_class","shots_lbl",metric_col]], out_csv)

            plt.figure(figsize=(12, 6))
            ax = sns.barplot(
                data=cur,
                x="shots_lbl", y=metric_col, hue="model",
                order=present, dodge=True, errorbar="ci"
            )
            ax.set_title(f"Few-shot Macro-F1 ({metric_title}) — Task: {tsk}")
            ax.set_ylabel("Macro-F1 (TEST)")
            ax.set_xlabel("Shots per class")
            plt.xticks(rotation=0)

            ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="Model", frameon=False)

            _bolden_axis(ax)
            plt.tight_layout()

            out_name = f"bar_fewshot_{metric_key}_{tsk}.png"
            plt.savefig(os.path.join(COMBINED_PLOTS_OUT, out_name), dpi=200)
            plt.close()

print(f"[OK] Few-shot per-task barplots (Linear & k-NN) saved to: {COMBINED_PLOTS_OUT}")

# --- Label-efficiency AUC (Linear) across shots per task ---
shot_x_map = {1:1, 5:5, 10:10, 25:25, 0:50}  # 0 ("All") -> large x
df_auc = []
for (model, task), g in df_fs.groupby(["model", "task"]):
    gg = g.copy()
    gg["x"] = gg["shots_per_class"].map(shot_x_map)
    gg = gg.dropna(subset=["x", "test_macroF1_linear"]).sort_values("x")
    if gg["x"].nunique() >= 2:
        x = gg["x"].values.astype(float)
        y = gg["test_macroF1_linear"].values.astype(float)
        x_norm = (x - x.min()) / (x.max() - x.min())
        # Guard against zero division (shouldn't happen because nunique>=2)
        denom = (x_norm.max() - x_norm.min()) if (x_norm.max() - x_norm.min()) > 0 else 1.0
        auc = np.trapz(y, x_norm) / denom
        df_auc.append({"model": model, "task": task, "auc_linear": auc})
df_auc = pd.DataFrame(df_auc)

if len(df_auc):
    # Save per-task AUC values
    save_df_csv(df_auc, os.path.join(COMBINED_PLOTS_OUT, "PLOT_DATA_label_efficiency_auc_linear_by_task.csv"))

    plt.figure(figsize=(12, 6))
    auc_bar = df_auc.groupby("model", as_index=False)["auc_linear"].mean()
    # Save aggregated AUC means per model
    save_df_csv(auc_bar, os.path.join(COMBINED_PLOTS_OUT, "PLOT_DATA_label_efficiency_auc_linear_mean_over_tasks.csv"))

    sns.barplot(data=auc_bar, x="model", y="auc_linear")
    plt.title("Label-Efficiency AUC (Linear Probe) — Mean over tasks")
    plt.ylabel("AUC (0–1)"); plt.xlabel("")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.savefig(os.path.join(COMBINED_PLOTS_OUT, "bar_label_efficiency_auc_linear.png"), dpi=200)
    plt.close()

# --- Best-of (Linear vs k-NN) at All-train ---
df_alltrain = bench[bench["shots_per_class"] == 0].copy()
if len(df_alltrain):
    df_alltrain["best_macroF1"] = df_alltrain[["test_macroF1_linear", "test_macroF1_knn"]].max(axis=1)
    # Save raw + best-of
    save_df_csv(df_alltrain, os.path.join(COMBINED_PLOTS_OUT, "PLOT_DATA_bestof_alltrain_raw.csv"))
    save_df_csv(df_alltrain[["model","task","best_macroF1"]], os.path.join(COMBINED_PLOTS_OUT, "PLOT_DATA_bestof_alltrain_compact.csv"))

    plt.figure(figsize=(14, 6))
    order_tasks = [t for t in TASKS if t in df_alltrain["task"].unique()]
    sns.barplot(
        data=df_alltrain, x="task", y="best_macroF1", hue="model",
        order=order_tasks, errorbar="ci", dodge=True
    )
    plt.title("Best-of (Linear or k-NN) Macro-F1 — TEST, All-train")
    plt.ylabel("Macro-F1"); plt.xlabel("Task")
    plt.xticks(rotation=30, ha="right")
    plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="Model")
    plt.tight_layout()
    plt.savefig(os.path.join(COMBINED_PLOTS_OUT, "bar_bestof_alltrain.png"), dpi=200)
    plt.close()

# --- Win-rate across tasks (All-train, Linear) ---
if len(df_alltrain):
    wr_rows = []
    for tsk, g in df_alltrain.groupby("task"):
        g2 = g.sort_values("test_macroF1_linear", ascending=False)
        if not g2.empty:
            top_val = g2["test_macroF1_linear"].iloc[0]
            winners = g2[g2["test_macroF1_linear"] >= top_val - 1e-9]["model"].unique()
            for w in winners:
                wr_rows.append({"model": w, "win_task": tsk})
    df_wr = pd.DataFrame(wr_rows)

    if len(df_wr):
        # Save raw win tasks + win counts
        save_df_csv(df_wr, os.path.join(COMBINED_PLOTS_OUT, "PLOT_DATA_winrate_tasks_linear_alltrain.csv"))
        winrate = df_wr.groupby("model")["win_task"].nunique().reset_index()
        winrate = winrate.rename(columns={"win_task": "win_count"})
        save_df_csv(winrate, os.path.join(COMBINED_PLOTS_OUT, "PLOT_DATA_winrate_counts_linear_alltrain.csv"))

        plt.figure(figsize=(10, 5))
        sns.barplot(data=winrate, x="model", y="win_count")
        plt.title("Win-rate (#tasks won) — Linear, All-train")
        plt.ylabel("#Tasks won"); plt.xlabel("")
        plt.xticks(rotation=20, ha="right")
        plt.tight_layout()
        plt.savefig(os.path.join(COMBINED_PLOTS_OUT, "bar_winrate_alltrain.png"), dpi=200)
        plt.close()

# --- k-robustness: lower variance across k is better ---
if len(ks_all):
    var_df = ks_all.groupby(["model", "task"])["test_macroF1_knn"].std(ddof=0).reset_index()
    var_df = var_df.rename(columns={"test_macroF1_knn": "std_over_k"})
    # Save robustness data
    save_df_csv(var_df, os.path.join(COMBINED_PLOTS_OUT, "PLOT_DATA_knn_robustness_std_over_k.csv"))

    plt.figure(figsize=(12, 6))
    sns.barplot(data=var_df, x="task", y="std_over_k", hue="model")
    plt.title("k-NN Robustness — Std of Macro-F1 over k (lower is better)")
    plt.ylabel("Std(F1) over k"); plt.xlabel("Task")
    plt.xticks(rotation=30, ha="right")
    plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="Model")
    plt.tight_layout()
    plt.savefig(os.path.join(COMBINED_PLOTS_OUT, "bar_knn_robustness_std_over_k.png"), dpi=200)
    plt.close()

# 5) (Optional) Table preview in stdout: Macro-F1 (Linear, All-train)
if len(df_bar):
    print("\n=== Macro-F1 (TEST, Linear, All-train) by task & model ===")
    pivot = df_bar.pivot_table(index="task", columns="model", values="test_macroF1_linear", aggfunc="mean")
    print(pivot.round(3).to_string())
    # Save the pivot too
    save_df_csv(pivot.reset_index(), os.path.join(COMBINED_PLOTS_OUT, "TABLE_macroF1_linear_alltrain_pivot.csv"))

print(f"\n[OK] Combined plots + CSVs saved under: {COMBINED_PLOTS_OUT}")

In [ ]:
import os
import re
import pandas as pd

# =======================================================
# Paths
# =======================================================
ROOT = os.path.join("baseline_eval_combined", "plots")
CSV_PATH = os.path.join(ROOT, "ALL_downstream_results_merged.csv")
OUT_TEX = os.path.join(ROOT, "downstream_results_tables.tex")

# =======================================================
# Load merged downstream results
# =======================================================
if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"CSV not found:\n{CSV_PATH}")

bench = pd.read_csv(CSV_PATH)

required_cols = {
    "model", "task", "shots_per_class",
    "test_macroF1_linear", "test_macroF1_knn"
}
missing = required_cols - set(bench.columns)
if missing:
    raise ValueError(f"Missing required columns in CSV: {missing}")

# Task ordering
TASKS = list(bench["task"].dropna().unique())

# Float format: ALWAYS 3 decimals
float_fmt = "{0:.3f}".format

latex_output = ""

# Small helper to wrap a LaTeX tabular in \resizebox{\textwidth}{!}{...}
def wrap_resizebox(tabular_str: str) -> str:
    return "\\resizebox{\\textwidth}{!}{%\n" + tabular_str + "}\n"

# =======================================================
# 1) ALL-TRAIN TABLES (shots_per_class == 0)
# =======================================================
df_alltrain = bench[bench["shots_per_class"] == 0].copy()

if not df_alltrain.empty:
    if "task" in df_alltrain.columns:
        df_alltrain["task"] = pd.Categorical(df_alltrain["task"], categories=TASKS, ordered=True)
        df_alltrain = df_alltrain.sort_values("task")

    # ---------------- Linear probe ----------------
    pivot_lin = df_alltrain.pivot_table(
        index="task",
        columns="model",
        values="test_macroF1_linear",
        aggfunc="mean"
    )

    # Ensure 3 decimals, but we rely on float_format in to_latex
    col_fmt_lin = "l" + "r" * pivot_lin.shape[1]

    lin_tabular = pivot_lin.to_latex(
        index=True,
        escape=True,
        column_format=col_fmt_lin,
        na_rep="--",
        float_format=float_fmt
    )

    latex_output += (
        "\\begin{table}[ht]\n"
        "\\centering\n"
        "\\scriptsize\n"
        "\\setlength{\\tabcolsep}{6pt}\n"
        "\\renewcommand{\\arraystretch}{1.2}\n"
        + wrap_resizebox(lin_tabular) +
        "\\caption{Macro-F1 (TEST, Linear probe) across downstream tasks using all training samples.}\n"
        "\\label{tab:downstream_linear_alltrain}\n"
        "\\end{table}\n\n"
    )

    # ---------------- k-NN ----------------
    pivot_knn = df_alltrain.pivot_table(
        index="task",
        columns="model",
        values="test_macroF1_knn",
        aggfunc="mean"
    )

    col_fmt_knn = "l" + "r" * pivot_knn.shape[1]

    knn_tabular = pivot_knn.to_latex(
        index=True,
        escape=True,
        column_format=col_fmt_knn,
        na_rep="--",
        float_format=float_fmt
    )

    latex_output += (
        "\\begin{table}[ht]\n"
        "\\centering\n"
        "\\scriptsize\n"
        "\\setlength{\\tabcolsep}{6pt}\n"
        "\\renewcommand{\\arraystretch}{1.2}\n"
        + wrap_resizebox(knn_tabular) +
        "\\caption{Macro-F1 (TEST, $k$-NN) across downstream tasks using all training samples.}\n"
        "\\label{tab:downstream_knn_alltrain}\n"
        "\\end{table}\n\n"
    )

# =======================================================
# 2) FEW-SHOT TABLES (1, 5, 10, 25, All)
#    For both Linear & k-NN, per task
# =======================================================
df_fs = bench.copy()
if not df_fs.empty:
    df_fs["shots_lbl"] = df_fs["shots_per_class"].replace({0: "All"}).astype(str)
    desired_order = ["1", "5", "10", "25", "All"]

    METRICS = [
        ("test_macroF1_linear", "Linear probe", "linear"),
        ("test_macroF1_knn",    "k-NN",         "knn"),
    ]

    for metric_col, metric_title, metric_key in METRICS:
        if metric_col not in df_fs:
            continue

        for tsk in TASKS:
            cur = df_fs[df_fs["task"] == tsk].copy()
            if cur.empty:
                continue

            cur["shots_lbl"] = pd.Categorical(
                cur["shots_lbl"],
                categories=desired_order,
                ordered=True
            )
            cur = cur.sort_values("shots_lbl")

            pivot_fs = cur.pivot_table(
                index="shots_lbl",
                columns="model",
                values=metric_col,
                aggfunc="mean"
            ).dropna(how="all")

            if pivot_fs.empty:
                continue

            col_fmt_fs = "l" + "r" * pivot_fs.shape[1]

            fs_tabular = pivot_fs.to_latex(
                index=True,
                escape=True,
                column_format=col_fmt_fs,
                na_rep="--",
                float_format=float_fmt
            )

            safe_task = re.sub(r"[^A-Za-z0-9]+", "_", tsk).strip("_")

            latex_output += (
                "\\begin{table}[ht]\n"
                "\\centering\n"
                "\\scriptsize\n"
                "\\setlength{\\tabcolsep}{6pt}\n"
                "\\renewcommand{\\arraystretch}{1.2}\n"
                + wrap_resizebox(fs_tabular) +
                f"\\caption{{Few-shot Macro-F1 (TEST, {metric_title}) for task {tsk}.}}\n"
                f"\\label{{tab:downstream_fewshot_{metric_key}_{safe_task}}}\n"
                "\\end{table}\n\n"
            )

# =======================================================
# Write .tex to file
# =======================================================
os.makedirs(ROOT, exist_ok=True)
with open(OUT_TEX, "w", encoding="utf-8") as f:
    f.write(latex_output)

print(f"[OK] LaTeX tables saved to:\n{OUT_TEX}")

### Confusion matrices

In [ ]:
# =====================================================
# DOWNSTREAM EVALUATION + CONFUSION MATRICES
# - Evaluates embeddings with linear probe & k-NN
# - Saves per-sample predictions
# - Creates publication-ready confusion matrices per task
# =====================================================

import os, copy, re, glob, json
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import ParameterGrid

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# -------------------------
# Global config
# -------------------------
SEED = 6740
np.random.seed(SEED)

FAST_MODE = True                       # flip False for full evaluation
FEW_SHOT_GRID = [1, 5, 10, 25, None]   # None means "All train"
K_GRID = [1, 5, 10, 20, 50]
K_FOR_KNN = 10 if FAST_MODE else 20
C_GRID = [0.1, 1.0] if FAST_MODE else [0.01, 0.1, 0.5, 1.0, 2.0, 5.0]
MAX_ITER_LR = 800 if FAST_MODE else 2000
MIN_CLASS_COUNT = 100 if FAST_MODE else 50
PCA_DIM = None                         # e.g., 256 to speed up; None disables
METABOFM_VERSION = "last"

FEATS_PRIMARY = "image_feats.npy"
FEATS_ALT_LAST = "image_feats_last.npy"
INDEX_FILE = "index.csv"
REQUIRED_INDEX = INDEX_FILE

# -------------------------
# Paths / run root
# -------------------------

SPLIT_CSV    = os.path.join("splits_by_dataset_id.csv")
IDX_PARQUET  = "metaspace_images_dump/msi_fm_samples.parquet"
MAN_PARQUET  = "metaspace_images_dump/manifest_expanded.parquet"

BENCH_OUT = os.path.join("baseline_eval")
PLOTS_OUT = os.path.join(BENCH_OUT, "plots")
os.makedirs(BENCH_OUT, exist_ok=True)
os.makedirs(PLOTS_OUT, exist_ok=True)

# NEW: combined benchmark output (multi-model, multi-run)
COMBINED_BENCH_OUT = os.path.join("baseline_eval_combined")
COMBINED_PLOTS_OUT = os.path.join(COMBINED_BENCH_OUT, "plots")
os.makedirs(COMBINED_PLOTS_OUT, exist_ok=True)

# -------------------------
# Embedding sets to evaluate
# -------------------------
EMB_SETS = {
    "msi_multitask_dinov2_untrained":    os.path.join("pretrained_feats2", "msi_multitask_dinov2_untrained"),
    "msi_multitask_mae_untrained":    os.path.join("pretrained_feats2", "msi_multitask_mae_untrained"),
     "msi_multitask_dinov2":    os.path.join("pretrained_feats2", "msi_multitask_dinov2"),
    "msi_multitask_mae":    os.path.join("pretrained_feats2", "msi_multitask_mae")
}

# -------------------------
# Canonicalization / merging
# -------------------------
def _clean(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    s = re.sub(r"\s+", " ", s)
    return s

def canonicalize_labels(df):
    df = df.copy()

    # 1) Polarity
    pol_map = {
        "pos": "Positive", "positive": "Positive", "+": "Positive",
        "neg": "Negative", "negative": "Negative", "-": "Negative"
    }

    def canon_polarity(s):
        if s is None:
            return None
        t = _clean(s).lower()
        t2 = pol_map.get(t, t)
        if t2 in ("positive", "negative"):
            return t2.capitalize()
        if "pos" in t:
            return "Positive"
        if "neg" in t:
            return "Negative"
        return _clean(s)

    if "polarity" in df.columns:
        df["polarity"] = df["polarity"].map(canon_polarity)

    # 2) Ionisation Source
    def canon_ion_src(s):
        if s is None:
            return None
        t_raw = _clean(s)
        t = t_raw.upper().replace("-", "").replace("_", "")
        if "APSMALDI" in t:
            return "AP-SMALDI"
        if "IRMALDESI" in t or "IRMALDI" in t:
            return "IR-MALDESI"
        if "APMALDI" in t:
            return "AP-MALDI"
        if "DESIMSI" in t:
            return "DESI"
        if "DESI" in t:
            return "DESI"
        if "MALDI" in t:
            return "MALDI"
        return t_raw

    if "ionisationSource" in df.columns:
        df["ionisationSource"] = df["ionisationSource"].map(canon_ion_src)

    # 3) Analyzer Type
    def canon_analyzer(s):
        if s is None:
            return None
        t = _clean(s)
        tl = t.lower()
        if "timstof" in tl and "flex" in tl:
            return "timsTOF Flex"
        if "fticr" in tl:
            if "12t" in tl:
                return "12T FTICR"
            if "7t" in tl and "scimax" in tl:
                return "FTICR scimaX 7T"
            return "FTICR"
        if "orbitrap" in tl or "q-exactive" in tl:
            return "Orbitrap"
        if "tof" in tl and "reflector" in tl:
            return "TOF reflector"
        if tl.strip() == "qtof":
            return "qTOF"
        return t

    if "analyzerType" in df.columns:
        df["analyzerType"] = df["analyzerType"].map(canon_analyzer)

    # 4) Organism
    def canon_organism(s):
        if s is None:
            return None
        t = _clean(s)
        tl = t.lower()
        if "|" in t or "," in t:
            if ("human" in tl or "homo sapiens" in tl) and (
                "mouse" in tl or "mus musculus" in tl
            ):
                return "Mixed"
        if "homo sapiens" in tl or tl.strip() in {"human", "h. sapiens", "homo"}:
            return "Homo sapiens"
        if "mus musculus" in tl or tl.strip() in {"mouse", "m. musculus"}:
            return "Mus musculus"
        return t

    if "organism" in df.columns:
        df["organism"] = df["organism"].map(canon_organism)

    # 5) Organism_Part
    def canon_part(s):
        if s is None:
            return None
        t = _clean(s)
        tl = t.lower()
        if "kidney" in tl:
            return "Kidney"
        if "brain" in tl:
            return "Brain"
        if "liver" in tl:
            return "Liver"
        if "lung" in tl:
            return "Lung"
        if "breast" in tl:
            return "Breast"
        if "skin" in tl:
            return "Skin"
        if "heart" in tl or "cardiac" in tl:
            return "Heart"
        return t

    if "Organism_Part" in df.columns:
        df["Organism_Part"] = df["Organism_Part"].map(canon_part)

    # 6) Condition
    def canon_condition(s):
        if s is None:
            return None
        t = _clean(s)
        tl = t.lower()
        if tl in {"n/a", "na", "none", "not available", ""}:
            return "NA"
        if tl in {"biopsy", "biopsies"}:
            return "Biopsy"
        if "fresh frozen" in tl or "frozen" in tl:
            return "Frozen"
        if "tumor" in tl or "tumour" in tl:
            return "Tumor"
        if "cancer" in tl:
            return "Cancer"
        if "wildtype" in tl or tl == "wt":
            return "Wildtype"
        if "healthy" in tl or "control" in tl:
            return "Healthy"
        if "diseased" in tl or "disease" in tl:
            return "Diseased"
        return t

    if "Condition" in df.columns:
        df["Condition"] = df["Condition"].map(canon_condition)

    return df

# -------------------------
# Load metadata + splits
# -------------------------
idx = pd.read_parquet(IDX_PARQUET)
man = pd.read_parquet(MAN_PARQUET)

need_cols = [
    "dataset_id",
    "organism",
    "polarity",
    "Organism_Part",
    "Condition",
    "analyzerType",
    "ionisationSource",
]
man_sub = man[[c for c in need_cols if c in man.columns]].drop_duplicates("dataset_id")

df_meta = idx.merge(man_sub, on="dataset_id", how="left", suffixes=("", "_man"))
df_meta = df_meta.loc[:, ~df_meta.columns.duplicated()].copy().reset_index(drop=True)
splits = pd.read_csv(SPLIT_CSV)
df_meta = df_meta.merge(splits, on="dataset_id", how="left")

if df_meta.duplicated("sample_path").sum():
    print("[WARN] duplicate sample_path rows in metadata; keeping first.")
    df_meta = df_meta.drop_duplicates("sample_path", keep="first").reset_index(drop=True)

df_meta = canonicalize_labels(df_meta)

# -------------------------
# Helpers
# -------------------------
TASKS = [
    "organism",
    "polarity",
    "Organism_Part",
    "Condition",
    "analyzerType",
    "ionisationSource",
]
EXCLUDE_LABELS = {"Condition": {"NA"}}

def get_feats_path(model_tag: str, emb_dir: str) -> str | None:
    if model_tag == "metabofm":
        fname = FEATS_PRIMARY if METABOFM_VERSION == "best" else FEATS_ALT_LAST
        path = os.path.join(emb_dir, fname)
        if os.path.exists(path):
            print(f"[INFO] Using MetaboFM-{METABOFM_VERSION.upper()} embeddings: {path}")
            return path
        else:
            print(f"[WARN] MetaboFM-{METABOFM_VERSION.upper()} embeddings not found at {path}")
            return None
    else:
        path = os.path.join(emb_dir, FEATS_PRIMARY)
        return path if os.path.exists(path) else None

def embeddings_ready(model_tag: str, emb_dir: str) -> bool:
    feats_path = get_feats_path(model_tag, emb_dir)
    return feats_path is not None and os.path.exists(os.path.join(emb_dir, REQUIRED_INDEX))

def resolve_model_outdir(model_tag: str, emb_dir: str) -> str:
    if model_tag == "metabofm":
        return TRAIN_OUT
    return emb_dir

def filter_valid(df_task, yname, min_count=5):
    x = df_task.dropna(subset=[yname]).copy()
    if yname in EXCLUDE_LABELS:
        x = x[~x[yname].isin(EXCLUDE_LABELS[yname])]
    x = x[x[yname].astype(str).str.len() > 0]
    vc = x[yname].value_counts()
    keep = vc[vc >= min_count].index
    x = x[x[yname].isin(keep)].copy()
    return x

def few_shot_subset(df_task, yname, shots_per_class=None, seed=SEED):
    if not shots_per_class or shots_per_class <= 0:
        return (df_task["split"] == "train").values
    rng = np.random.RandomState(seed)
    m_train = (df_task["split"] == "train").values
    keep = np.zeros(len(df_task), dtype=bool)
    labels = df_task.loc[m_train, yname].astype(str).values
    idxs = np.where(m_train)[0]
    from collections import defaultdict
    per_class = defaultdict(list)
    for i, lbl in zip(idxs, labels):
        per_class[lbl].append(i)
    for lbl, arr in per_class.items():
        arr = np.array(arr)
        rng.shuffle(arr)
        keep[arr[: min(shots_per_class, len(arr))]] = True
    return keep

def get_mask(df_all, split_name):
    return (df_all["split"] == split_name).values

def run_linear_probe(X_tr, y_tr, X_va, y_va, X_te, y_te):
    pipe = Pipeline(
        [
            ("scaler", StandardScaler(with_mean=True, with_std=True)),
            (
                "clf",
                LogisticRegression(
                    solver="saga",
                    max_iter=MAX_ITER_LR,
                    class_weight="balanced",
                    random_state=SEED,
                    n_jobs=-1,
                ),
            ),
        ]
    )
    grid = {"clf__C": C_GRID}
    best = None
    best_va = -np.inf
    for p in tqdm(list(ParameterGrid(grid)), desc="LinearProbe grid", leave=False):
        pipe.set_params(**p)
        pipe.fit(X_tr, y_tr)
        pred_va = pipe.predict(X_va)
        macro_f1 = f1_score(y_va, pred_va, average="macro")
        if macro_f1 > best_va:
            best_va = macro_f1
            best = copy.deepcopy(pipe)
    pred_te = best.predict(X_te)
    acc = accuracy_score(y_te, pred_te)
    f1m = f1_score(y_te, pred_te, average="macro")
    return acc, f1m, pred_te, best

def run_knn(X_tr, y_tr, X_te, y_te, k=20):
    n_fit = int(X_tr.shape[0])
    if n_fit < 1 or len(np.unique(y_tr)) < 2:
        return np.nan, np.nan, np.array([], dtype=object), None, 0
    k_eff = max(1, min(k, n_fit))
    knn = KNeighborsClassifier(n_neighbors=k_eff, metric="cosine", n_jobs=-1)
    knn.fit(X_tr, y_tr)
    if X_te.shape[0] == 0:
        return np.nan, np.nan, np.array([], dtype=object), knn, k_eff
    pred_te = knn.predict(X_te)
    acc = accuracy_score(y_te, pred_te)
    f1m = f1_score(y_te, pred_te, average="macro")
    return acc, f1m, pred_te, knn, k_eff

def per_class_f1(y_true, y_pred):
    rep = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    out = {
        k: v["f1-score"]
        for k, v in rep.items()
        if k not in ("accuracy", "macro avg", "weighted avg")
    }
    return out

# -------------------------
# Evaluate ONE embedding set dir (saves per-sample preds)
# -------------------------
def evaluate_embeddings(emb_dir: str, model_tag: str):
    feats_path = get_feats_path(model_tag, emb_dir)
    index_path = os.path.join(emb_dir, INDEX_FILE)
    if not (feats_path and os.path.exists(feats_path) and os.path.exists(index_path)):
        print(
            f"[WARN] Missing feats or index for {model_tag} at {emb_dir}; "
            f"looked for {FEATS_PRIMARY}"
            f"{' or ' + FEATS_ALT_LAST if model_tag=='metabofm' else ''} and {INDEX_FILE}. Skipping."
        )
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    if os.path.basename(feats_path) == FEATS_ALT_LAST:
        print(f"[INFO] ({model_tag}) Using LAST embeddings: {feats_path}")
    else:
        print(f"[INFO] ({model_tag}) Using BEST embeddings: {feats_path}")

    emb = np.load(feats_path)
    index = pd.read_csv(index_path)  # must include 'sample_path'
    index = index.reset_index().rename(columns={"index": "row_id"})

    df_base = df_meta.copy()
    df_emb = df_base.merge(index, on="sample_path", how="inner")
    df_emb = df_emb.sort_values("row_id").reset_index(drop=True)

    if emb.shape[0] != len(df_emb):
        print(
            f"[INFO] ({model_tag}) Embedding count != joined rows; aligning to matched rows only."
        )
        emb = emb[df_emb["row_id"].values, :]
    if emb.shape[0] != len(df_emb):
        raise RuntimeError(
            f"({model_tag}) Embedding count and joined rows mismatch after alignment."
        )

    if PCA_DIM is not None and PCA_DIM > 0 and PCA_DIM < emb.shape[1]:
        print(f"[INFO] ({model_tag}) Reducing dim to {PCA_DIM} via PCA.")
        from sklearn.decomposition import PCA

        pca = PCA(n_components=PCA_DIM, random_state=SEED)
        emb = pca.fit_transform(emb)

    df_emb["row_pos"] = np.arange(len(df_emb), dtype=int)

    results = []
    pcs_rows = []
    kgrid_rows = []
    pred_rows = []  # per-sample predictions

    for yname in tqdm(TASKS, desc=f"Tasks[{model_tag}]"):
        if yname not in df_emb.columns:
            print(f"[WARN] ({model_tag}) Missing column {yname}; skipping.")
            continue

        print(f"\n==== [{model_tag}] Task: {yname} ====")
        df_task = filter_valid(df_emb, yname, min_count=MIN_CLASS_COUNT)
        if df_task.empty:
            print(f"[WARN] ({model_tag}) Skipping {yname}: no data after filtering.")
            continue
        if df_task["split"].isna().any():
            df_task = df_task[~df_task["split"].isna()].copy()

        row_pos = df_task["row_pos"].values
        X = emb[row_pos]
        y = df_task[yname].astype(str).values
        m_va = get_mask(df_task, "val")
        m_te = get_mask(df_task, "test")

        def _ok(mask):
            return np.sum(mask) > 0 and (len(np.unique(y[mask])) > 1)

        for shots in FEW_SHOT_GRID:
            m_tr = few_shot_subset(df_task, yname, shots_per_class=shots)
            if not (_ok(m_tr) and _ok(m_va) and _ok(m_te)):
                print(
                    f"[WARN] ({model_tag}) {yname} few-shot={shots}: insufficient classes."
                )
                continue

            acc_lp, f1_lp, pred_lp, best_lp = run_linear_probe(
                X[m_tr], y[m_tr], X[m_va], y[m_va], X[m_te], y[m_te]
            )
            acc_knn, f1_knn, pred_knn, _, k_eff = run_knn(
                X[m_tr], y[m_tr], X[m_te], y[m_te], k=K_FOR_KNN
            )

            results.append(
                {
                    "model": model_tag,
                    "task": yname,
                    "shots_per_class": shots if shots else 0,
                    "test_acc_linear": acc_lp,
                    "test_macroF1_linear": f1_lp,
                    "test_acc_knn": acc_knn,
                    "test_macroF1_knn": f1_knn,
                    "k_for_knn": int(k_eff),
                    "pca_dim": PCA_DIM if PCA_DIM else X.shape[1],
                    "n_train": int(m_tr.sum()),
                    "n_val": int(m_va.sum()),
                    "n_test": int(m_te.sum()),
                    "n_classes": int(len(np.unique(y))),
                }
            )

            # Per-class & per-sample only for All-train (shots=None)
            if shots is None:
                # Per-class F1
                pcs = per_class_f1(y[m_te], pred_lp)
                for cls, f1v in pcs.items():
                    pcs_rows.append(
                        {
                            "model": model_tag,
                            "task": yname,
                            "label": cls,
                            "f1": f1v,
                            "shots_per_class": 0,
                        }
                    )

                # Per-sample predictions for confusion matrices
                te_idx = np.where(m_te)[0]
                df_task_te = df_task.iloc[te_idx].copy()
                for j, (true_lbl, p_lp, p_knn) in enumerate(
                    zip(y[m_te], pred_lp, pred_knn)
                ):
                    row = df_task_te.iloc[j]
                    base_info = {
                        "model": model_tag,
                        "task": yname,
                        "shots_per_class": 0,
                        "sample_path": row.get("sample_path", None),
                        "dataset_id": row.get("dataset_id", None),
                        "split": row.get("split", None),
                        "y_true": str(true_lbl),
                    }
                    pred_rows.append(
                        {**base_info, "clf_type": "linear", "y_pred": str(p_lp)}
                    )
                    pred_rows.append(
                        {**base_info, "clf_type": "knn", "y_pred": str(p_knn)}
                    )

            # k-sweep (All-train only)
            if shots is None:
                for k in K_GRID:
                    acc_k, f1_k, _, _, k_eff_k = run_knn(
                        X[m_tr], y[m_tr], X[m_te], y[m_te], k=k
                    )
                    if np.isnan(f1_k):
                        continue
                    kgrid_rows.append(
                        {
                            "model": model_tag,
                            "task": yname,
                            "k": int(k_eff_k),
                            "test_macroF1_knn": f1_k,
                        }
                    )

    res_df = pd.DataFrame(results).sort_values(
        ["task", "model", "shots_per_class"]
    )
    pcs_df = (
        pd.DataFrame(pcs_rows).sort_values(["task", "model", "label"])
        if len(pcs_rows)
        else pd.DataFrame()
    )
    kgrid_df = (
        pd.DataFrame(kgrid_rows).sort_values(["task", "model", "k"])
        if len(kgrid_rows)
        else pd.DataFrame()
    )
    preds_df = pd.DataFrame(pred_rows) if len(pred_rows) else pd.DataFrame()

    primary_out_dir = resolve_model_outdir(model_tag, emb_dir)
    os.makedirs(primary_out_dir, exist_ok=True)

    res_df.to_csv(os.path.join(primary_out_dir, "downstream_results.csv"), index=False)
    if len(pcs_df):
        pcs_df.to_csv(
            os.path.join(primary_out_dir, "per_class_f1_linear.csv"), index=False
        )
    if len(kgrid_df):
        kgrid_df.to_csv(
            os.path.join(primary_out_dir, "knn_k_sweep.csv"), index=False
        )
    if len(preds_df):
        preds_df.to_csv(
            os.path.join(primary_out_dir, "per_sample_predictions.csv"), index=False
        )

    print(f"[OK] ({model_tag}) Saved per-model results to: {primary_out_dir}")

    mirror_dir = os.path.join(BENCH_OUT, model_tag)
    os.makedirs(mirror_dir, exist_ok=True)
    res_df.to_csv(os.path.join(mirror_dir, "downstream_results.csv"), index=False)
    if len(pcs_df):
        pcs_df.to_csv(
            os.path.join(mirror_dir, "per_class_f1_linear.csv"), index=False
        )
    if len(kgrid_df):
        kgrid_df.to_csv(os.path.join(mirror_dir, "knn_k_sweep.csv"), index=False)
    if len(preds_df):
        preds_df.to_csv(
            os.path.join(mirror_dir, "per_sample_predictions.csv"), index=False
        )

    return res_df, pcs_df, kgrid_df

# -------------------------
# RUN EVAL FOR ALL MODELS
#   (skip if per_sample_predictions already exist)
# -------------------------
all_main, all_pcs, all_ks = [], [], []
for tag, emb_dir in EMB_SETS.items():
    if not embeddings_ready(tag, emb_dir):
        print(
            f"[SKIP] '{tag}': embeddings missing at {emb_dir} "
            f"(need {FEATS_PRIMARY} or {FEATS_ALT_LAST}, plus {INDEX_FILE})."
        )
        continue

    primary_out_dir = resolve_model_outdir(tag, emb_dir)
    primary_pred = os.path.join(primary_out_dir, "per_sample_predictions.csv")
    mirror_dir   = os.path.join(BENCH_OUT, tag)
    mirror_pred  = os.path.join(mirror_dir, "per_sample_predictions.csv")

    # If predictions already exist, don't recompute — but ensure mirror exists
    if os.path.exists(mirror_pred) or os.path.exists(primary_pred):
        print(f"[SKIP] '{tag}': per-sample predictions already exist.")
        if not os.path.exists(mirror_dir):
            os.makedirs(mirror_dir, exist_ok=True)
        if not os.path.exists(mirror_pred) and os.path.exists(primary_pred):
            # Sync from primary to mirror once
            pd.read_csv(primary_pred).to_csv(mirror_pred, index=False)
        continue

    print(f"\n[RUN] Evaluating model '{tag}' from {emb_dir}")
    res_df, pcs_df, kgrid_df = evaluate_embeddings(emb_dir, tag)
    if len(res_df):
        all_main.append(res_df)
    if len(pcs_df):
        all_pcs.append(pcs_df)
    if len(kgrid_df):
        all_ks.append(kgrid_df)

# =====================================================
# CONFUSION MATRICES (publication ready) FROM PREDICTIONS
# =====================================================

mpl.rcParams["svg.fonttype"] = "none"
sns.set(style="white")

# Save confusion matrices to COMBINED_PLOTS_OUT
CM_OUT_DIR = os.path.join(COMBINED_PLOTS_OUT, "confusion_matrices")
os.makedirs(CM_OUT_DIR, exist_ok=True)

TITLE_FS = 18
LABEL_FS = 16
TICK_FS = 13
CBAR_FS = 13

def _safe_filename(s: str) -> str:
    s = str(s).strip()
    s = re.sub(r"[^\w\-_.]+", "_", s)
    return s

def _plot_confusion_matrix(
    cm,
    classes,
    normalize="true",
    title="Confusion matrix",
    cmap="Blues",
    out_base=None,
    show_counts=True,
):
    cm = np.array(cm, dtype=float)
    cm_disp = cm.copy()

    if normalize == "true":
        row_sums = cm_disp.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1.0
        cm_disp = cm_disp / row_sums
    elif normalize == "pred":
        col_sums = cm_disp.sum(axis=0, keepdims=True)
        col_sums[col_sums == 0] = 1.0
        cm_disp = cm_disp / col_sums
    elif normalize == "all":
        total = cm_disp.sum()
        if total == 0:
            total = 1.0
        cm_disp = cm_disp / total

    fig, ax = plt.subplots(figsize=(8, 7))

    hm = sns.heatmap(
        cm_disp,
        annot=False,
        cmap=cmap,
        cbar=True,
        square=True,
        xticklabels=classes,
        yticklabels=classes,
        ax=ax,
    )

    cbar = hm.collections[0].colorbar
    cbar.ax.tick_params(labelsize=CBAR_FS)
    cbar.set_label(
        "Proportion" if normalize else "Count",
        fontsize=CBAR_FS,
        fontweight="bold",
    )

    ax.set_xticklabels(
        ax.get_xticklabels(),
        rotation=35,
        ha="right",
        fontsize=TICK_FS,
        fontweight="bold",
    )
    ax.set_yticklabels(
        ax.get_yticklabels(), rotation=0, fontsize=TICK_FS, fontweight="bold"
    )

    ax.set_xlabel("Predicted label", fontsize=LABEL_FS, fontweight="bold")
    ax.set_ylabel("True label", fontsize=LABEL_FS, fontweight="bold")
    ax.set_title(title, fontsize=TITLE_FS, fontweight="bold", pad=12)

    n_classes = len(classes)
    for i in range(n_classes):
        for j in range(n_classes):
            if normalize:
                val = cm_disp[i, j] * 100.0
                text = f"{val:4.1f}%"
                if show_counts:
                    cnt = int(cm[i, j])
                    text = f"{cnt}\n{val:4.1f}%"
            else:
                cnt = int(cm[i, j])
                text = f"{cnt}"
            ax.text(
                j + 0.5,
                i + 0.5,
                text,
                ha="center",
                va="center",
                fontsize=10,
                color="black",
                fontweight="bold",
            )

    plt.tight_layout()
    if out_base is not None:
        png_path = out_base + ".png"
        fig.savefig(png_path, dpi=300)
        print(f"[OK] Saved confusion matrix: {png_path}")
    plt.close(fig)

# 1) Load all per_sample_predictions from BENCH_OUT/*/
pred_files = glob.glob(os.path.join(BENCH_OUT, "*", "per_sample_predictions.csv"))
if not pred_files:
    raise FileNotFoundError(
        f"No per_sample_predictions.csv found under {BENCH_OUT}. "
    )

df_list = []
for pf in pred_files:
    try:
        df_list.append(pd.read_csv(pf))
        print(f"[INFO] Loaded predictions from {pf}")
    except Exception as e:
        print(f"[WARN] Failed to load {pf}: {e}")

df_pred = pd.concat(df_list, ignore_index=True)
print(f"[INFO] Total prediction rows loaded: {len(df_pred)}")

# 2) Filter to All-train, TEST split, linear probe
df_pred = df_pred[
    (df_pred["shots_per_class"] == 0)
    & (df_pred["split"] == "test")
    & (df_pred["clf_type"] == "linear")
].copy()

required_cols = {"model", "task", "y_true", "y_pred"}
missing = required_cols.difference(df_pred.columns)
if missing:
    raise ValueError(f"Missing required columns in predictions: {missing}")

df_pred["y_true"] = df_pred["y_true"].astype(str)
df_pred["y_pred"] = df_pred["y_pred"].astype(str)

# (Optional) Save merged per-sample predictions to combined plots dir
merged_preds_path = os.path.join(COMBINED_PLOTS_OUT, "ALL_per_sample_predictions_merged.csv")
df_pred.to_csv(merged_preds_path, index=False)

# 3) One confusion matrix per (model, task) with task-specific labels
for (model_name, task_name), g in df_pred.groupby(["model", "task"]):
    y_true = g["y_true"].values
    y_pred = g["y_pred"].values

    if len(np.unique(y_true)) <= 1:
        print(f"[WARN] Skipping {model_name} / {task_name}: only one class in TEST.")
        continue

    # Task-specific label set
    labels = sorted(set(y_true).union(set(y_pred)))

    cm = confusion_matrix(y_true, y_pred, labels=labels)

    title = f"Confusion matrix — {task_name} — {model_name}"
    safe_model = _safe_filename(model_name)
    safe_task = _safe_filename(task_name)
    out_base = os.path.join(CM_OUT_DIR, f"cm_{safe_task}_{safe_model}")

    _plot_confusion_matrix(
        cm=cm,
        classes=labels,
        normalize="true",
        title=title,
        cmap="Blues",
        out_base=out_base,
        show_counts=True,
    )

print(f"\n[OK] All confusion matrices written to: {CM_OUT_DIR}")